# Ouroboros Quickstart (Gemini & iFlow Edition)

A self-modifying AI agent that writes its own code and evolves autonomously.

**Before running:**

1. [Fork the repository](https://github.com/joi-lab/ouroboros/fork) on GitHub
2. Add your API keys in the **Secrets** sidebar (key icon on the left):
   - `IFLOW_API_KEY` — required (from https://platform.iflow.cn)
   - `TELEGRAM_BOT_TOKEN` — required ([@BotFather](https://t.me/BotFather))
   - `TOTAL_BUDGET` — required, spending limit in USD (e.g. `50`)
   - `GITHUB_TOKEN` — required ([github.com/settings/tokens](https://github.com/settings/tokens), `repo` scope)
   - `GOOGLE_API_KEY` — required (for Gemini 2.0 Pro)
   - `TAVILY_API_KEY` — optional, for better web search
3. **Check `GITHUB_USER`** in the cell below (it's set to Aisrefot-Reed)
4. Run the cell (Shift+Enter)
5. Open your Telegram bot and send any message — you become the owner

In [ ]:
import os
import shutil
from google.colab import userdata

# ⚠️ Configuration for Aisrefot-Reed
CFG = {
    "GITHUB_USER": "Aisrefot-Reed",
    "GITHUB_REPO": "ouroboros",
    
    # PRIMARY MODEL: Gemini 2.0 Pro (Premium)
    "OUROBOROS_MODEL": "google/gemini-2.0-pro-exp-02-05", 
    
    # CODE MODEL: Qwen3-Coder-Plus
    "OUROBOROS_MODEL_CODE": "Qwen3-Coder-Plus",
    
    # LIGHT MODEL: Qwen3-Coder-30B-A3B-Instruct
    "OUROBOROS_MODEL_LIGHT": "Qwen3-Coder-30B-A3B-Instruct",
    
    # WEBSEARCH MODEL: Gemini Thinking
    "OUROBOROS_WEBSEARCH_MODEL": "google/gemini-2.0-flash-thinking-exp-01-21",
    
    "OUROBOROS_MAX_WORKERS": "5",
    "OUROBOROS_MAX_ROUNDS": "200",
    "OUROBOROS_BG_BUDGET_PCT": "15",
}

# 1. Auto-load secrets from Colab panel
SECRETS = [
    "IFLOW_API_KEY", "TELEGRAM_BOT_TOKEN", "TOTAL_BUDGET", 
    "GITHUB_TOKEN", "GOOGLE_API_KEY", "OPENAI_API_KEY", "TAVILY_API_KEY"
]

print("--- Loading Secrets ---")
for s in SECRETS:
    try:
        val = userdata.get(s)
        os.environ[s] = str(val)
        print(f"✅ {s} loaded")
    except:
        if s in ["IFLOW_API_KEY", "TELEGRAM_BOT_TOKEN", "TOTAL_BUDGET", "GITHUB_TOKEN", "GOOGLE_API_KEY"]:
            print(f"❌ ERROR: Secret {s} not found!")
        else:
            print(f"⚠️ {s} not set (optional)")

# 2. Apply configuration
for k, v in CFG.items():
    os.environ[k] = str(v)

# 3. Prepare repository
%cd /content
if os.path.exists("/content/ouroboros_repo"):
    print("Cleaning up old repository...")
    shutil.rmtree("/content/ouroboros_repo")

repo_url = f"https://github.com/{CFG['GITHUB_USER']}/{CFG['GITHUB_REPO']}.git"
print(f"Cloning {repo_url}...")
!git clone {repo_url} /content/ouroboros_repo
%cd /content/ouroboros_repo

# 4. Install dependencies & browser
print("Installing system components...")
!pip install -q -r requirements.txt
!pip install -q google-generativeai playwright-stealth
!playwright install chromium
!playwright install-deps chromium

# 5. Run
print("Launching Ouroboros...")
%run colab_bootstrap_shim.py